# VTM parcial no Taskonomy

Notebook fino sobre o pacote `vtm`. Ative uma GPU em **Runtime → Change runtime type** antes do treino.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

A célula seguinte clona este repositório do GitHub na primeira execução e executa `git pull` nas sessões seguintes.

In [ ]:
from pathlib import Path
import subprocess

repo = Path('/content/Visual-Token-Matching')
if not (repo / '.git').exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/NataLira1/Visual-Token-Matching.git',
        str(repo),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)

%cd /content/Visual-Token-Matching
!pip install -q -e ".[experiment]"
import torch, timm
print('torch', torch.__version__, 'timm', timm.__version__, 'gpu', torch.cuda.get_device_name(0))

## Download seletivo

Esta etapa valida a instalação do `omnidata-tools==0.0.23` e substitui sua integração obsoleta com Google Forms por um aceite explícito no próprio notebook. Os links oficiais das licenças são exibidos e nenhum nome/e-mail é transmitido. O download e o pré-processamento não fazem parte das quatro horas de GPU.

In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y aria2
import importlib

# O pacote também contém um módulo legado chamado
# omnidata_tools.starter_dataset que não funciona. O downloader usa este:
from omnidata_tools.dataset import starter_dataset
downloader = importlib.import_module('omnidata_tools.dataset.download')

required_licenses = {'omnidata', 'taskonomy'}
missing_licenses = required_licenses - starter_dataset.STARTER_DATA_LICENSES.keys()
if missing_licenses:
    raise RuntimeError(
        'Instalação incompatível do omnidata-tools; licenças ausentes: '
        + ', '.join(sorted(missing_licenses))
    )
print('Instalação do omnidata-tools validada.')

print('\nLeia as licenças antes de continuar:')
for component in ('omnidata', 'taskonomy'):
    print(f'- {component}: {starter_dataset.STARTER_DATA_LICENSES[component]}')
license_acceptance = input(
    "\nDigite ACEITO para confirmar que leu e aceita os termos acima: "
).strip()
if license_acceptance != 'ACEITO':
    raise RuntimeError('Termos não aceitos; download cancelado.')

# A implementação original tenta registrar o aceite em um Google Form antigo,
# que atualmente encerra o CLI com código 1. O aceite acima é explícito e a
# substituição abaixo evita transmitir dados pessoais para esse endpoint.
def local_license_confirmation(components, require_prompt, component_to_license, email, name):
    for component in sorted(set(components) | {'omnidata'}):
        if component not in component_to_license:
            raise RuntimeError(f'Licença ausente para {component}.')
    print('Licenças aceitas explicitamente nesta execução do notebook.')

downloader.licenses_clickthrough = local_license_confirmation
download_options = dict(
    domains=['rgb', 'segment_semantic'],
    components=['taskonomy'],
    subset='tiny',
    split='all',
    dest='/content/drive/MyDrive/taskonomy',
    dest_compressed='/content/drive/MyDrive/taskonomy_compressed',
    connections_total=16,
    n_workers=4,
    agree_all=True,
)

print('\nExecutando dry-run...')
# keep_compressed=True evita outro bug da versão 0.0.23, que tenta apagar
# arquivos simulados pelo dry-run e produz FileNotFoundError.
downloader.download(**download_options, dryrun=True, keep_compressed=True)
dry_run_ok = True
print('Dry-run validado. A próxima célula pode ser executada.')

In [ ]:
from pathlib import Path

if not globals().get('dry_run_ok', False):
    raise RuntimeError('Execute e valide primeiro a célula de dry-run.')

downloader.download(**download_options, dryrun=False)

data_root = Path('/content/drive/MyDrive/taskonomy')
downloaded_files = [path for path in data_root.rglob('*') if path.is_file()]
if not downloaded_files:
    raise RuntimeError(
        'O downloader retornou sucesso, mas nenhum arquivo foi encontrado no destino.'
    )
print(f'Download validado: {len(downloaded_files)} arquivos encontrados.')
for path in downloaded_files[:10]:
    print(path)

## Preparação, meta-treino e avaliação

In [ ]:
from pathlib import Path
import yaml
config = yaml.safe_load(Path('configs/taskonomy_vtm.yaml').read_text())
config['data']['root'] = '/content/drive/MyDrive/taskonomy'
config['data']['manifest'] = '/content/drive/MyDrive/vtm_outputs/taskonomy_manifest.json'
config['train']['checkpoint'] = '/content/drive/MyDrive/vtm_outputs/vtm_best.pt'
config['experiment']['output_dir'] = '/content/drive/MyDrive/vtm_outputs/evaluation'
runtime_config = Path('/content/vtm_taskonomy_runtime.yaml')
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
runtime_config

In [ ]:
!vtm-taskonomy prepare --config /content/vtm_taskonomy_runtime.yaml
!vtm-taskonomy train --config /content/vtm_taskonomy_runtime.yaml
!vtm-taskonomy evaluate --config /content/vtm_taskonomy_runtime.yaml

In [ ]:
import pandas as pd
from IPython.display import display, Image
out = Path(config['experiment']['output_dir'])
display(pd.read_csv(out / 'summary.csv'))
display(pd.read_json(out / 'hypothesis.json'))
panels = sorted((out / 'panels').glob('*.png'))
if panels:
    display(Image(filename=str(panels[0])))